# Astra 1.6B — V4 Training (bring your own dataset, 20,000 steps)

This notebook performs **full model distillation generation using the Gemini API** (Gemini 3.5 Flash-Lite) and then trains the student model for **20,000 steps**.
It includes API key setup (via Colab Secrets), concurrent teacher generation, audit/capping/splits, tokenizer preparation, and GPU training.

In [ ]:
import os, subprocess

REPO = '/content/astra'
REPO_URL = 'https://github.com/anoneurx/astra.git'
DATA_DIR = 'datasets/distillation_v3'
# Checkpoints live outside the repo so `git status` never sees them. Overridable so this
# notebook also runs outside Colab, where /content may not be writable.
RUNS = os.environ.get('ASTRA_RUNS', '/content/runs')
os.makedirs(RUNS, exist_ok=True)
DRIVE_ROOT = '/content/drive/MyDrive/astra/distillation_v3'

if not os.path.isdir(os.path.join(REPO, 'tools')):
    subprocess.run('git clone --depth 1 %s %s' % (REPO_URL, REPO), shell=True, check=True)
os.chdir(REPO)
assert os.path.isdir(os.path.join(REPO, 'tools')), 'astra clone missing'

try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    os.makedirs(DRIVE_ROOT, exist_ok=True)
    DRIVE_OK = os.path.isdir(DRIVE_ROOT)
except Exception as exc:
    DRIVE_OK = False
    print('Google Drive unavailable (%s) - artifacts stay under /content only.' % exc)

os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs('/content/runs', exist_ok=True)

print('repo        :', os.getcwd())
print('data dir    :', os.path.abspath(DATA_DIR))
print('drive mirror:', DRIVE_ROOT if DRIVE_OK else 'DISABLED')

## Stage 1 — Upload the generated dataset

The generator notebook (`astra_v4_data_generator.ipynb`) produced `astra_v4_records.zip` and
downloaded it. This cell unpacks that zip here into `datasets/distillation_v3/records.jsonl`
(plus `progress.json`, which only matters if you generate again) and verifies the checksum.

If you would rather place the file directly: drag `records.jsonl` into the Files sidebar and
skip to Stage 2 — the split stage reads `datasets/distillation_v3/records.jsonl`.


In [ ]:
import os, json, glob, hashlib, shutil, zipfile

DATA_DIR = 'datasets/distillation_v3'
os.makedirs(DATA_DIR, exist_ok=True)
RECORDS = os.path.join(DATA_DIR, 'records.jsonl')
UPLOAD_DIR = '/content'


def find_candidates():
    """Every place the records might already be, in one list.

    A file can reach the runtime three ways: the picker below writes to /content, a
    sidebar drag-drop lands in /content or the working directory, and a Drive mount is a
    third possibility. Scanning all of them means this cell does not care which route the
    file took. records.jsonl is matched by name; a zip has to look like a records archive,
    so an unrelated zip in the runtime cannot be mistaken for the data.
    """
    found, seen = [], set()
    for d in ('.', UPLOAD_DIR, DATA_DIR):
        for pat in ('*.zip', 'records.jsonl'):
            for p in sorted(glob.glob(os.path.join(d, pat))):
                full = os.path.abspath(p)
                if full in seen or full == os.path.abspath(RECORDS):
                    continue
                base = os.path.basename(p)
                if pat == '*.zip' and not any(
                        k in base for k in ('astra_v4_records', 'records', 'distillation')):
                    continue
                if os.path.getsize(p) <= 0:
                    continue
                seen.add(full)
                found.append(p)
    return found


def unpack(src):
    """Put records.jsonl where the rest of the pipeline expects it.

    A bare records.jsonl is copied rather than extracted, so the picker accepts either
    shape. A zip must carry records.jsonl at its root: nested one level down, extraction
    silently lands it in a subdirectory and every later stage reads an empty file.
    """
    if not src.endswith('.zip'):
        shutil.copy2(src, RECORDS)
        print('copied %s -> %s' % (src, RECORDS))
        return
    with zipfile.ZipFile(src) as zf:
        names = zf.namelist()
        if 'records.jsonl' not in names:
            raise RuntimeError(
                'the zip at %s has no records.jsonl at its root (it holds %s). Rebuild it '
                'so records.jsonl sits at the top level.' % (src, names))
        zf.extract('records.jsonl', DATA_DIR)
        if 'progress.json' in names:
            zf.extract('progress.json', DATA_DIR)
    print('extracted records.jsonl ->', RECORDS)


cands = find_candidates()

# An existing records.jsonl wins over any archive: never clobber what a previous run made.
if os.path.exists(RECORDS) and os.path.getsize(RECORDS) > 0:
    print('records.jsonl already present (%.1f MB) - keeping it.' % (os.path.getsize(RECORDS) / 1e6))
elif not cands:
    # Nothing staged, so ask for it here instead of sending the user to the sidebar
    # uploader. files.upload() hands back {name: bytes} in memory, which keeps a 5.5 MB
    # archive out of git and off a download URL.
    print('No records archive found yet - a file picker opens next.')
    print('Choose astra_v4_records.zip (5.5 MB). A bare records.jsonl works too.')
    try:
        from google.colab import files as colab_files
    except ImportError:
        colab_files = None
    if colab_files is None:
        raise SystemExit('No file picker outside Colab. Put astra_v4_records.zip or '
                         'records.jsonl in %s and re-run this cell.' % UPLOAD_DIR)
    for name, blob in colab_files.upload().items():
        dest = os.path.join(UPLOAD_DIR, os.path.basename(name))
        with open(dest, 'wb') as fh:
            fh.write(blob)
        print('uploaded %s -> %s (%.1f MB)' % (name, dest, len(blob) / 1e6))
    cands = find_candidates()
    if not cands:
        raise SystemExit('nothing usable came back from the picker - '
                         'pick astra_v4_records.zip, or records.jsonl on its own')
    unpack(cands[0])
else:
    print('found archive:', cands[0])
    if len(cands) > 1:
        print('  also present, ignored: %s' % ', '.join(cands[1:]))
    unpack(cands[0])

assert os.path.exists(RECORDS), 'no records.jsonl - upload astra_v4_records.zip and re-run this cell'

n, bad = 0, 0
with open(RECORDS, encoding='utf-8') as fh:
    for line in fh:
        line = line.strip()
        if not line:
            continue
        try:
            json.loads(line); n += 1
        except json.JSONDecodeError:
            bad += 1
print('validated %d records (%d unreadable lines)' % (n, bad))
if bad:
    raise RuntimeError('%d unreadable lines in %s - fix before training' % (bad, RECORDS))

print()
print('digest of %s:' % RECORDS)
print('  sha256  ', hashlib.sha256(open(RECORDS, 'rb').read()).hexdigest())
print('  size    %.1f MB' % (os.path.getsize(RECORDS) / 1e6))
print('Compare the sha256 with the one printed by the generator notebook to confirm')
print('the upload is byte-identical to what you generated.')

## Stage 2 — Class balance, splits, manifest

**Class cap first.** Any single `classification` is capped at `MAX_CLASS_SHARE` of the
corpus, with a floor of `MIN_PER_CLASS` so capping can never starve a class out of the
training split. Dropping is deterministic — sorted on `dedup_key`, then a seeded sample — so
the same corpus always balances the same way. The before/after counts and everything dropped
go into `manifest.json` under `class_balance`, so the corpus is self-describing.

**`records.jsonl` keeps every paid row, including the ones the cap dropped.** The cap is
applied to the in-memory corpus that becomes the splits, not to the file. That is deliberate:
the teacher rows are the expensive part, so keeping them means you can re-balance later with
a different `MAX_CLASS_SHARE` without paying for generation again. Read the class balance
from `manifest.json` or from the split files — `records.jsonl` is the uncapped input, and
counting classes there will show the skew the cap exists to correct. Expect roughly 40% of
paid rows to be discarded at the v2 skew of 67/21/12; the generator is set to 120k records to
leave a large balanced corpus, and raising the generator’s `TARGET_SAMPLES` is the only lever for more.

Then stratified splits, and the row-level disjointness proof: the split unit is one teacher
record, so a leaked record would show up as a shared body across two splits. That count must
be zero in every pair.


In [ ]:
import os, json, time, random, hashlib, shutil, subprocess
from collections import Counter

REPO = '/content/astra'
if not os.path.isdir(os.path.join(REPO, 'tools')):
    subprocess.run('git clone --depth 1 https://github.com/anoneurx/astra.git %s' % REPO,
                   shell=True, check=True)
os.chdir(REPO)

DATA_DIR = 'datasets/distillation_v3'
RECORDS = os.path.join(DATA_DIR, 'records.jsonl')
DRIVE_ROOT = '/content/drive/MyDrive/astra/distillation_v3'
SEED, VAL_FRAC, EVAL_FRAC = 20051, 0.01, 0.04
MIN_INPUT_CHARS, MIN_ANALYSIS_CHARS = 120, 40
# These caps bound a row. They do NOT make an exchange fit the 256-token block, and the
# comment that used to claim they did was wrong. Measured on the salvaged corpus at
# 3.17 chars/token, a 256-token block holds ~812 chars for the input AND the target
# together, while 480 + ~500 leaves ~980. 31% of blocks still overflow. Tightening input
# alone cannot fix it - at input<=240 it is still 6.6% over, because the target is the
# binding side. What the cap does buy is a worst case: max block fell 421 -> 380 tokens
# versus V1's 1200 cap, and a runaway teacher is rejected at generation time rather than
# discovered in the audit. The pre-flight cell reports the real distribution; read it
# before assuming a corpus fits.
MAX_INPUT_CHARS, MAX_ANALYSIS_CHARS = 480, 420
# A teacher that loops will happily emit the same six events fifty times. That is valid JSON,
# passes every enum check and is worse than useless as training data, so reject it: if any
# 60-character span of the row occurs three or more times, the generation degenerated.
# Calibrated against every corpus in datasets/ - zero false positives over 7958 chat blocks.
REPEAT_ANCHOR, MAX_REPEATS = 60, 3
# A flag the model can actually learn: anything actionable is an early warning.
EARLY_WARNING_CLASSIFICATIONS = ('suspicious', 'malicious')
MAX_CONFIDENCE = 0.97
MIN_ACTION_CHARS = 40
# Phrases that assert there is nothing to do. Measured on V1: 122 of 1002 targets were under
# 40 characters and the single most common was "No action required." 93 times.
NO_OP_ACTIONS = ('no action required', 'no action needed', 'none', 'no action',
                 'no further action', 'nothing', 'n/a', 'not applicable')

# Class cap and floor. Previously inherited from the teacher-config cell, which is gone: this
# notebook now expects an uploaded corpus, so the balance stage owns its own knobs.
# Any single classification is capped at MAX_CLASS_SHARE of the corpus, with a floor of
# MIN_PER_CLASS so capping can never starve a class out of the training split.
MAX_CLASS_SHARE = 0.45
MIN_PER_CLASS = 40

CATEGORIES = ['Normal behavior', 'Suspicious behavior', 'Early-warning sequences',
              'False positives', 'Multi-signal correlation', 'Authentication',
              'Network', 'Process behavior', 'Vulnerability / code security']

try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    DRIVE_OK = os.path.isdir(DRIVE_ROOT)
except Exception:
    DRIVE_OK = False

assert os.path.exists(RECORDS), 'no records.jsonl - run Stage 1 (upload) first'
records = [json.loads(l) for l in open(RECORDS, encoding='utf-8') if l.strip()]
assert records, 'records.jsonl is empty'
print('loaded %d records' % len(records))


def to_chat_block(rec):
    """datasets/chat/make_corpus.py:273 format."""
    return 'You: %s\nAstra: %s Classification: %s, risk %s. Recommended action: %s' % (
        rec['input'], rec['analysis'], rec['classification'].upper(),
        rec['risk'], rec['recommended_action'])


def to_alert_line(rec, row_id):
    """tools/build_cyber_corpus.py:52 shape, prose telemetry instead of 41 NSL-KDD features."""
    return 'ROW#%d ALERT %s | EARLY_WARNING %s RISK=%s HARDNESS=%d\n' % (
        row_id, rec['input'], rec['classification'], rec['risk'], rec['hardness'])


# ---- class balance cap -------------------------------------------------------
# Applied here, after generation, because it is the only point where the realised
# distribution is observable. The category weights in Stage 1 look like they should do this
# and do not: they are prompt archetypes, and the teacher picks the label.
#
# Two things this gets wrong if written naively, both measured on real data:
#
#  * A one-shot threshold does not hold. Capping against the ORIGINAL total leaves the
#    result above the target, because deleting rows lowers the total and the target with
#    it. 562 rows at 67% suspicious capped once to 253 leaves 57.6% against a 45% target.
#  * Iterating to a fixed point overshoots into the floor. As the total shrinks the cap
#    shrinks with it, and the MIN_PER_CLASS floor then dominates: a 995-row corpus with a
#    5-row rare class converged to 47% by cutting the two large classes to 40 each, keeping
#    85 of 995 rows. A single-class corpus collapsed 500 -> 40, which is data loss, not
#    balance - with one class there is no skew to correct.
#
# So: solve for the largest cap that satisfies the constraint, by binary search, and treat
# "no cap satisfies it" as "do not touch the data". _share() is monotone in K, so the
# search is exact rather than iterative.
def _capped(counts, K, floor):
    return {c: min(n, max(floor, K)) for c, n in counts.items()}


def _share(counts, K, floor):
    capped = _capped(counts, K, floor)
    return max(capped.values()) / max(1, sum(capped.values()))


before = Counter(r['classification'] for r in records)
dropped = Counter()
after = Counter()
n_before = sum(before.values())
cap = None
if len(before) > 1:
    lo, hi = 0, max(before.values())
    while lo <= hi:
        mid = (lo + hi) // 2
        if _share(before, mid, MIN_PER_CLASS) <= MAX_CLASS_SHARE + 1e-12:
            cap = mid
            lo = mid + 1
        else:
            hi = mid - 1

if cap is None:
    print('class balance: no cap satisfies %.0f%% for this distribution - left alone'
          % (MAX_CLASS_SHARE * 100))
    print('  a corpus with a single classification has no skew to correct.')
    print('  one whose rare classes are already at the MIN_PER_CLASS floor cannot be')
    print('  balanced by discarding data. Cutting rows here would lose training signal.')
    after = Counter(before)
    records = list(records)
else:
    rng = random.Random(SEED)
    by_class = {}
    for rec in records:
        by_class.setdefault(rec['classification'], []).append(rec)
    for items in by_class.values():
        items.sort(key=lambda r: r['dedup_key'])
    keep = {}
    for cls, items in sorted(by_class.items()):
        quota = min(len(items), max(MIN_PER_CLASS, cap))
        if quota >= len(items):
            keep[cls] = list(range(len(items)))
        else:
            order = list(range(len(items)))
            rng.shuffle(order)          # unbiased, and seeded so it repeats exactly
            keep[cls] = sorted(order[:quota])
        dropped[cls] = len(items) - len(keep[cls])
    records = [rec for c in by_class for i, rec in enumerate(by_class[c]) if i in keep[c]]
    after = Counter(r['classification'] for r in records)
    top = max(after.values()) / max(1, len(records))
    print('class balance: cap %d rows, max class %.1f%% of %d (was %.1f%% of %d), target %.0f%%'
          % (cap, top * 100, len(records),
             max(before.values()) / max(1, n_before) * 100, n_before, MAX_CLASS_SHARE * 100))
    for cls in sorted(set(before) | set(after)):
        print('  %-11s %5d -> %5d  (%5.1f%% -> %5.1f%%)'
              % (cls, before[cls], after[cls],
                 100.0 * before[cls] / n_before, 100.0 * after[cls] / max(1, len(records))))
    if dropped:
        print('  dropped %d over-represented rows: %s'
              % (sum(dropped.values()), {k: v for k, v in sorted(dropped.items()) if v}))
    assert records, 'the class cap removed every record'

by_cat = {}
for rec in records:
    by_cat.setdefault(rec['category'], []).append(rec)

splits = {'train': [], 'val': [], 'eval': []}
for cat, items in by_cat.items():
    items = sorted(items, key=lambda r: r['dedup_key'])
    n = len(items)
    n_val = min(n - 2, max(1, round(n * VAL_FRAC))) if n >= 3 else 0
    n_eval = (min(n - 2 - n_val, max(1, round(n * EVAL_FRAC)))
              if n - 2 - n_val >= 1 else 0)
    ordered = items[n_val + n_eval:] + items[:n_val] + items[n_val:n_val + n_eval]
    splits['train'].extend(ordered[: n - n_val - n_eval])
    splits['val'].extend(ordered[n - n_val - n_eval: n - n_eval])
    splits['eval'].extend(ordered[n - n_eval:])

for name in splits:
    splits[name].sort(key=lambda r: (r['category'], r['dedup_key']))

CHAT_DIR, ALERT_DIR = os.path.join(DATA_DIR, 'chat'), os.path.join(DATA_DIR, 'alerts')
os.makedirs(CHAT_DIR, exist_ok=True)
os.makedirs(ALERT_DIR, exist_ok=True)

split_meta = {'chat': {}, 'alerts': {}}
for name, items in splits.items():
    chat_path = os.path.join(CHAT_DIR, name + '.txt')
    alert_path = os.path.join(ALERT_DIR, name + '.txt')
    with open(chat_path, 'w', encoding='utf-8') as fh:
        # datasets/chat/make_corpus.py writes "\n\n".join(blocks) with no trailing newline;
        # match it exactly so both corpora are byte-comparable
        fh.write('\n\n'.join(to_chat_block(r) for r in items))
    with open(alert_path, 'w', encoding='utf-8') as fh:
        # tools/build_cyber_corpus.py writes one "\n"-terminated line per row
        for i, rec in enumerate(items):
            fh.write(to_alert_line(rec, i))
    dist = dict(Counter(r['classification'] for r in items).most_common())
    for flavour, path in (('chat', chat_path), ('alerts', alert_path)):
        blob = open(path, 'rb').read()
        split_meta[flavour][name] = {
            'path': path,
            'sha256': hashlib.sha256(blob).hexdigest(),
            'bytes': len(blob),
            'rows': len(items),
            'classification_distribution': dist,
        }
    print('%-6s %5d rows -> chat + alerts' % (name, len(items)))

bodies = {name: {r['dedup_key'] for r in items} for name, items in splits.items()}
shared = {'%s|%s' % (a, b): len(bodies[a] & bodies[b])
          for a, b in (('train', 'val'), ('train', 'eval'), ('val', 'eval'))}
print('shared record bodies across splits:', shared)
assert shared['train|val'] == 0 and shared['train|eval'] == 0, 'record body leaked across splits'

manifest = {
    'name': 'astra-distillation-v2',
    'kind': 'synthetic',
    'pipeline': 'colab/Distliation/astra_v1_distillation_v3.ipynb',
    'source_ids': ['gemini-3.5-flash-lite'],
    'teacher': 'gemini-3.5-flash-lite (Gemini API, GA 2026-07-21)',
    'teacher_model': 'gemini-3.5-flash-lite',
    'license': 'synthetic - generated output, no third-party corpus redistributed',
    'created_at': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()),
    'documents': len(records),
    'category_distribution': {c: len(by_cat.get(c, [])) for c in CATEGORIES},
    'dedup_redundancy_ratio': round(
        1.0 - len(bodies['train']) / max(1, len(splits['train'])), 6),
    'contamination_checked': True,
    'cleaning_rules': [
        'json-strict-parse: fenced or truncated replies rejected, never written',
        'schema-validate: all 8 required fields present and typed',
        'enum-check: classification in {normal,suspicious,malicious,unknown}',
        'risk-uppercase: risk in {INFO,LOW,MEDIUM,HIGH,CRITICAL} per configs/astra_200m.json',
        'single-line: multi-line teacher fields collapsed so the chat framing stays one turn per line',
        'dedup: sha256 of normalized input, global across all records',
        'min-length: input >= 120 chars, analysis >= 40 chars',
        'hardness-derived: hardness computed from risk, never taken from the teacher',
    ],
    'filters': [
        {'id': 'distill-v2-validator',
         'params': {'min_input_chars': MIN_INPUT_CHARS,
                    'min_analysis_chars': MIN_ANALYSIS_CHARS}},
        {'id': 'distill-v2-dedup', 'params': {'key': 'sha256(normalized input)[:16]'}},
    ],
    'hash': {'algorithm': 'sha256', 'value': split_meta['chat']['train']['sha256']},
    'row_level_safety': {
        'shared_record_bodies_across_splits': shared,
        'note': 'the split unit is one teacher record; the n=13 token gate is reported '
                'separately by the pre-flight cell because the You:/Astra: framing template '
                'repeats in every chat block',
    },
    'chat_format': 'You: <question> / Astra: <answer>, blank-line separated '
                   '(datasets/chat/make_corpus.py:273)',
    'alerts_format': 'ROW#<n> ALERT <telemetry> | EARLY_WARNING <classification> '
                     'RISK=<RISK> HARDNESS=<n> - shape from tools/build_cyber_corpus.py:52 with '
                     'prose telemetry instead of 41 NSL-KDD features; never concatenate with '
                     'datasets/cyber/',
    'splits': split_meta,
}

manifest_path = os.path.join(DATA_DIR, 'manifest.json')
manifest['class_balance'] = {
    'max_share': MAX_CLASS_SHARE,
    'before': dict(before),
    'after': dict(after),
    'dropped': dict(dropped),
}
with open(manifest_path, 'w', encoding='utf-8') as fh:
    json.dump(manifest, fh, indent=2, sort_keys=True)
    fh.write('\n')
print()
print('manifest ->', manifest_path)
for flavour in split_meta:
    for name in split_meta[flavour]:
        m = split_meta[flavour][name]
        print('  %-7s %-5s %5d rows %10d bytes sha=%s'
              % (flavour, name, m['rows'], m['bytes'], m['sha256'][:16]))

if DRIVE_OK:
    # each split keeps its own Drive subdirectory: chat/ and alerts/ both contain
    # train.txt, val.txt and eval.txt, so mirroring them flat would overwrite one with the other
    for rel in ('manifest.json',
                'chat/train.txt', 'chat/val.txt', 'chat/eval.txt',
                'alerts/train.txt', 'alerts/val.txt', 'alerts/eval.txt'):
        src = manifest_path if rel == 'manifest.json' else os.path.join(DATA_DIR, rel)
        if not os.path.exists(src):
            continue
        dst = os.path.join(DRIVE_ROOT, rel)
        os.makedirs(os.path.dirname(dst), exist_ok=True)
        shutil.copy2(src, dst)
    print('mirrored manifest + splits to Drive ->', DRIVE_ROOT)
    for rel in ('chat/train.txt', 'alerts/train.txt'):
        print('  %-18s %10d bytes'
              % (rel, os.path.getsize(os.path.join(DRIVE_ROOT, rel))))

print()
print('first train chat block:')
print(open(os.path.join(CHAT_DIR, 'train.txt'), encoding='utf-8').read().split('\n\n')[0])
print()
print('first train alert line:')
print(open(os.path.join(ALERT_DIR, 'train.txt'), encoding='utf-8').readline().strip())

## Stage 3 — Emit real training configs

Writes `configs/astra_distill_v4_{chat,alerts}.json` with paths that point at the corpora
just written. `leak_check` is off with a reason: the n=13 token gate compares raw token
n-grams, and every block repeats the `You:`/`Astra:` framing, so it measures the template
rather than contamination. The row-level proof above is the real gate.


In [ ]:
import os, json, shutil, subprocess

REPO = '/content/astra'
if not os.path.isdir(os.path.join(REPO, 'tools')):
    subprocess.run('git clone --depth 1 https://github.com/anoneurx/astra.git %s' % REPO,
                   shell=True, check=True)
os.chdir(REPO)

DATA_DIR = 'datasets/distillation_v3'
TOK = 'tokenizer/artifacts/prose_chat_word.json'
DRIVE_ROOT = '/content/drive/MyDrive/astra/distillation_v3'
SEED = 20051

try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    DRIVE_OK = os.path.isdir(DRIVE_ROOT)
except Exception:
    DRIVE_OK = False

MODEL_BLOCK = {
    'vocab_size': 16384,
    'd_model': 2048,
    'n_layers': 24,
    'n_heads': 16,
    'd_head': 128,
    'd_ffn': 8192,
    'max_seq_len': 512,
    'rope_theta': 10000.0,
    'eps': 1e-06,
    'tie_embeddings': True,
}

# 50,000 steps over 19,000 rows is ~19 epochs, so read val_loss rather than assuming the last
# step is the best one. 20,000 steps was ~7.6 epochs, which is why this run is different in
# kind and not just in length: the model can memorise 38 scenario families, and the only
# signal that it has is val_loss turning up while train_loss keeps falling.
TRAINING_BLOCK = {
    'max_steps': 50000, 'peak_lr': 3e-4, 'min_lr': 1e-6,
    # 60 warmup steps was 0.3% of a 20k schedule and is 0.12% of a 50k one. Warmup exists to
    # spend the first steps at a low enough LR that the AdamW moments settle instead of
    # thrashing, and that need does not shrink when the schedule is stretched - 1% of steps
    # is the usual choice, so 500 here.
    'warmup_steps': 500,
    'batch_seq': 8, 'accum_steps': 1, 'optimizer': 'adamw', 'scheduler': 'cosine',
    'weight_decay': 0.05, 'grad_clip': 1.0, 'val_every': 200, 'val_shards': 1,
    # 3.3 GB weights-only per snapshot, previous one deleted, so this costs 6.6 GB steady
    # state. These are for eval and inspection, NOT for resuming.
    'save_every': 5000,
    # The resumable state: fp16 weights + fp32 AdamW moments, previous one deleted. 16.5 GB
    # steady state, 33 GB mid-write, ~12s to write uncompressed against ~12min compressed.
    # At 50k steps on one GPU that is 10-25 hours, and Colab reclaims preemptible runtimes
    # without warning, so the run needs somewhere to come back to. Without this the only
    # recoverable state is written after the final step and a crash at step 41,000 is a
    # restart from zero.
    'resume_every': 10000,
    'experiment_store': 'experiments/runs',
}

LEAK_NOTE = (
    'n-gram gate disabled for the same reason as configs/astra5m_word_chat.json: with a '
    'word-level tokenizer 13 tokens is a ~13-word window, and the You:/Astra: framing plus a '
    'shared risk vocabulary repeats across every block, so the gate reports overlap on a '
    'corpus with zero record-level leakage. Record-level disjointness is proven in '
    'datasets/distillation_v3/manifest.json (row_level_safety) and the raw n=13 numbers are '
    'printed by the pre-flight cell.'
)

CONFIGS = {
    'configs/astra_distill_v4_chat.json': {
        'data': {'train': DATA_DIR + '/chat/train.txt', 'val': DATA_DIR + '/chat/val.txt'},
        'out_dir': 'checkpoints/astra_distill_v4_chat',
    },
    'configs/astra_distill_v4_alerts.json': {
        'data': {'train': DATA_DIR + '/alerts/train.txt', 'val': DATA_DIR + '/alerts/val.txt'},
        'out_dir': 'checkpoints/astra_distill_v4_alerts',
    },
}

os.makedirs('configs', exist_ok=True)
for path, spec in CONFIGS.items():
    for key in ('train', 'val'):
        assert os.path.exists(spec['data'][key]), 'missing corpus: ' + spec['data'][key]
    cfg = {'model': MODEL_BLOCK, 'training': TRAINING_BLOCK, 'tokenizer': TOK,
           'data': spec['data'], 'seed': SEED,
           'safety': {'leak_check': False, 'note': LEAK_NOTE},
           'out_dir': spec['out_dir']}
    with open(path, 'w', encoding='utf-8') as fh:
        json.dump(cfg, fh, indent=2)
        fh.write('\n')
    print('wrote %s\n      train %s\n      val   %s' % (path, spec['data']['train'], spec['data']['val']))

if DRIVE_OK:
    for path in CONFIGS:
        shutil.copy2(path, os.path.join(DRIVE_ROOT, os.path.basename(path)))
    print('configs mirrored to Drive ->', DRIVE_ROOT)

## Stage 4 — Pre-flight: tokenizer, coverage, block lengths, leak report

Rebuilds the tokenizer with `--extra` if it is missing (it is gitignored, so a fresh clone
has none) and reports what actually matters:

* **OOV rate.** The number that decides whether a 6-layer model can learn this text at all.
* **Block-length distribution.** The student's block is 256 tokens, so an exchange that does
  not fit is split and the model never sees the input and the answer in the same context.
  This is **reported, not enforced**, and the caps in Stage 1 do not fix it — the target is
  the binding side, not the telemetry. Read the numbers before assuming the corpus fits.
* **Leak report.** The figure the disabled gate would have tripped on.


In [ ]:
import os, sys, json, time, subprocess

REPO = '/content/astra'
if not os.path.isdir(os.path.join(REPO, 'tools')):
    subprocess.run('git clone --depth 1 https://github.com/anoneurx/astra.git %s' % REPO,
                   shell=True, check=True)
os.chdir(REPO)
sys.path.insert(0, os.path.join(REPO, 'python'))

DATA_DIR = 'datasets/distillation_v3'
TOK_ARTIFACT = 'tokenizer/artifacts/prose_chat_word.json'

# Rebuild every time, and fold THIS corpus into the vocab. The previous line was
# `build_word_tokenizer.py --extra` with no paths, and --extra is nargs='*', so it folded
# nothing: the artifact was built on prose + v1 chat alone and this corpus then OOV'd at
# 7.3% (chat) and 12.3% (alerts) - one token in eight was <unk>, which is capacity spent
# learning the vocabulary instead of the decision. Measured on this corpus, passing the
# training text costs v1's corpora 0.4pp of coverage and buys back 7.8pp and 12.3pp here:
#
#   corpus                built on prose+v1     with this corpus folded in
#   datasets/prose            1.29%                        1.70%
#   datasets/chat (v1)        1.13%                        1.48%
#   v4 chat                   7.86%                        0.02%
#   v4 alerts                12.82%                        0.56%
#
# Always rebuilding rather than reusing a cached artifact is deliberate: an artifact carried
# over from an earlier v1 or v2 run encodes a vocabulary this corpus never sees, and the
# OOV line printed below is the only thing that would catch it.
V4_TRAIN = [os.path.join(DATA_DIR, p) for p in ('chat/train.txt', 'alerts/train.txt')]
for p in V4_TRAIN:
    assert os.path.exists(p), 'missing corpus for the tokenizer: ' + p
# Flush first: the builder writes to the real fd via subprocess, so a buffered print
# would surface after the builder's own output and make the cell look out of order.
print('building tokenizer over prose + v1 chat + this corpus (a few seconds)', flush=True)
subprocess.run([sys.executable, 'tools/build_word_tokenizer.py', '--extra', *V4_TRAIN],
               check=True)
assert os.path.exists(TOK_ARTIFACT), 'tokenizer artifact still missing'

from astra.safety import leak_check
from astra.tokenizer.word import WordLevel

tok = WordLevel.load(TOK_ARTIFACT)
unk_id = tok.special_names.index('<unk>')
print('tokenizer: %d tokens (%d words), min_frequency=%d, unk_id=%d'
      % (len(tok), tok.stats.get('num_words', 0), tok.min_frequency, unk_id))
print()

train_text, val_text = {}, {}
for corpus, store in (('chat/train.txt', train_text), ('chat/val.txt', val_text),
                      ('alerts/train.txt', train_text), ('alerts/val.txt', val_text)):
    path = os.path.join(DATA_DIR, corpus)
    assert os.path.exists(path), 'missing corpus: ' + path
    store[corpus] = open(path, encoding='utf-8').read()

for corpus, text in train_text.items():
    ids = tok.encode(text)
    oov = sum(1 for i in ids if i == unk_id)
    print('%-16s %8d chars %8d tokens  OOV=%5.2f%%  bytes/token=%.1f'
          % (corpus, len(text), len(ids), 100.0 * oov / max(1, len(ids)),
             len(text) / max(1, len(ids))))
print()

# Block-length report. The student's block is 256 tokens, so an exchange that does not fit is
# split and the model never sees the input and the answer in the same context. This is REPORTED,
# not enforced, and the reason is that the repo already tolerates it: datasets/chat sits at 49.8%
# over 256. Do not read a high number here as a bug to be fixed by lowering the input cap - on
# the salvaged corpus, input<=240 still leaves 6.6% of blocks over 256, because the target is the
# binding side. To actually fit 256 you have to shorten analysis and recommended_action, not the
# telemetry. V1 (1200 cap) peaked at 421 tokens; the 480 cap brings that to 380.
BLOCK_TOKENS = 256
blocks = [b for b in train_text['chat/train.txt'].split('\n\n') if b.strip()]
blen = sorted(len(tok.encode(b)) for b in blocks)
nblocks = len(blen)
print('chat block length (student block = %d tokens, %d blocks)' % (BLOCK_TOKENS, nblocks))
print('  tokens min/p50/p90/p95/p99/max : %d / %d / %d / %d / %d / %d'
      % (blen[0], blen[nblocks // 2], blen[int(nblocks * .90)], blen[int(nblocks * .95)],
         blen[int(nblocks * .99)], blen[-1]))
for lim in (256, 384, 512):
    over = sum(1 for x in blen if x > lim)
    print('  over %3d tokens : %5d / %5d  (%5.1f%%)'
          % (lim, over, nblocks, 100.0 * over / max(1, nblocks)))
print()

result = leak_check(tok.encode(train_text['chat/train.txt']),
                    tok.encode(val_text['chat/val.txt']), n=13)
print('leak_check(chat train, chat val, n=13) =')
print(json.dumps(result, indent=2))
print()
print('That is the number the disabled safety gate would have tripped on. The gate is off')
print('because a repeated chat framing template makes any overlap meaningless here; the')
print('record-level proof lives in manifest.json under row_level_safety.')

manifest_path = os.path.join(DATA_DIR, 'manifest.json')
assert os.path.exists(manifest_path), 'no manifest.json - run the materialize cell first'
manifest = json.load(open(manifest_path, encoding='utf-8'))
manifest['leak_report'] = result
manifest['preflight'] = {
    'tokenizer': TOK_ARTIFACT,
    'tokenizer_vocab_size': len(tok),
    'block_tokens': BLOCK_TOKENS,
    'chat_block_tokens': {
        'count': nblocks, 'min': blen[0], 'p50': blen[nblocks // 2],
        'p90': blen[int(nblocks * .90)], 'p95': blen[int(nblocks * .95)],
        'p99': blen[int(nblocks * .99)], 'max': blen[-1],
        'over_256': sum(1 for x in blen if x > 256),
        'over_384': sum(1 for x in blen if x > 384),
    },
    'recorded_at': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()),
}
with open(manifest_path, 'w', encoding='utf-8') as fh:
    json.dump(manifest, fh, indent=2, sort_keys=True)
    fh.write('\n')
print()
print('leak_report + preflight recorded in', manifest_path)

## Stage 5a — Training

On by default: this is the stage that turns a corpus into a model. Two stages — **alerts
first** (sequence + label head), then **chat warm-started from the alerts checkpoint** with
`--reset-step` so the LR schedule and step counter restart rather than resuming mid-schedule.

**Use an A100 for this run.** 50,000 steps is 10-25 hours on an A100 and roughly 60-100 hours
on a T4 at the same utilisation, and a T4 also has the smallest disk of the options (about
78 GB against 235 GB here), which is what the 16.5 GB resume state and the 6.6 GB of snapshots
are sized against. Set the runtime with Runtime → Change runtime type → A100, or the cell
asserts rather than falling back to CPU, because a CPU run would look like a hang rather than
an error.

**The run survives a disconnect.** Every 10,000 steps the trainer writes a resumable
checkpoint (weights + AdamW moments + step) and deletes the previous one. If the runtime is
reclaimed, just re-run this cell: it finds the newest resumable checkpoint and continues from
that step. It resumes the *alerts* stage, which is the long one; the chat stage is started
from the alerts final and is a fresh 50,000-step schedule by design.

`bf16` is used whenever torch reports the GPU as supporting it; a T4 reports support via
emulation, so a T4 run uses bf16. The check stays for older torch and for GPUs that genuinely
lack it. No GradScaler is needed on the fp16 path because `gpu_train.py` does not scale the
loss.

In [ ]:
import glob, json

RUN_TRAIN = True
DTYPE = 'bf16'
# Read from the config so this cell cannot disagree with what the trainer was told to do.
with open('configs/astra_distill_v4_alerts.json', encoding='utf-8') as _fh:
    MAX_STEPS = json.load(_fh)['training']['max_steps']


def run_logged(cmd, log):
    """Run a training command, streaming its output live but keeping a log.

    `subprocess.run(shell=True, check=True)` is what hid the real error the first time this
    ran: the CalledProcessError carried the command and not the message, so the actual
    failure only existed in a scrollback nobody read. Stream it AND keep the tail."""
    print('\n$ ' + cmd, flush=True)
    with open(log, 'w', encoding='utf-8') as fh:
        proc = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE,
                                stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in proc.stdout:
            sys.stdout.write(line)
            fh.write(line)
    rc = proc.wait()
    if rc:
        tail = ''.join(open(log, encoding='utf-8').readlines()[-40:])
        raise SystemExit('%s\n\nfailed with exit %d. Last 40 lines of %s:\n%s'
                         % (cmd, rc, log, tail))


def find_resume_point(out_dir):
    """The newest resumable checkpoint under a run directory, or None.

    gpu_train.py writes `resume-<step>.npz` (weights + AdamW + step) alongside the cheap
    weights-only `checkpoint-<step>.npz` snapshots. Only the first can continue a run, so
    this reads the manifest rather than globbing for a filename: the loader refuses a
    weights-only file on purpose, and a script that picked one would die with a confusing
    error several minutes into a restart.
    """
    hits = []
    for cand in glob.glob(os.path.join(out_dir, '**', 'resume-*.npz'), recursive=True):
        man = cand.rsplit('.npz', 1)[0] + '.manifest.json'
        if not os.path.exists(man):
            continue
        try:
            m = json.load(open(man))
        except ValueError:
            continue
        if m.get('optimizer_included') and m.get('step', 0) < MAX_STEPS:
            hits.append((m['step'], cand))
    return max(hits)[1] if hits else None


def _manifest(path):
    man = path.rsplit('.npz', 1)[0] + '.manifest.json'
    if not os.path.exists(man):
        return {}
    try:
        return json.load(open(man))
    except ValueError:
        return {}


def completed_final(out_dir, target_steps):
    """The newest final.npz that actually reached *target_steps*, or None.

    Newest by mtime, not by sort order. A resumed run writes into <out>/resumed/, so a
    directory can hold both <out>/final.npz from an earlier run and <out>/resumed/final.npz
    from a later one, and taking the alphabetically first would silently return the older
    model - the wrong weights, with no error anywhere.
    """
    hits = [p for p in glob.glob(os.path.join(out_dir, '**', 'final.npz'), recursive=True)
            if _manifest(p).get('step', 0) >= target_steps]
    return max(hits, key=os.path.getmtime) if hits else None


def find_final(out_dir):
    """gpu_train.py writes <out>/final.npz, or <out>/resumed/final.npz with --resume."""
    plain = os.path.join(out_dir, 'final.npz')
    if os.path.exists(plain):
        return plain
    hits = sorted(glob.glob(os.path.join(out_dir, '**', 'final.npz'), recursive=True))
    return hits[0] if hits else None


def persist(out_dir):
    if not DRIVE_OK:
        return
    dst = os.path.join(DRIVE_ROOT, 'runs', os.path.basename(out_dir))
    os.makedirs(dst, exist_ok=True)
    for src in [os.path.join(out_dir, n) for n in ('final.npz', 'report.json')] + \
            glob.glob(os.path.join(out_dir, '*.manifest.json')):
        if os.path.exists(src):
            shutil.copy2(src, os.path.join(dst, os.path.basename(src)))


import torch
if not RUN_TRAIN:
    print('RUN_TRAIN is False - set it to True to spend GPU time.')
else:
    assert torch.cuda.is_available(), (
        'No GPU visible. Runtime -> Change runtime type -> T4, then re-run this cell. '
        'Falling back to CPU here would look like a hang rather than an error.')
    gpu = torch.cuda.get_device_name(0)
    print('torch %s  cuda=%s  %s' % (torch.__version__, torch.cuda.is_available(), gpu))
    if DTYPE == 'bf16' and not torch.cuda.is_bf16_supported():
        cap = torch.cuda.get_device_capability(0)
        print('%s has no native bfloat16 (sm_%d) - torch.amp.autocast raises on it, '
              'so using fp16' % (gpu, cap[0] * 10 + cap[1]))
        DTYPE = 'fp16'
    print('dtype:', DTYPE)

    out_alerts = os.path.join(RUNS, 'distill_v4_alerts')
    os.makedirs(out_alerts, exist_ok=True)
    # A 50k-step alerts run is many hours on one GPU, and Colab takes preemptible runtimes
    # back without warning. If a resumable checkpoint is on disk, continue from it instead of
    # training the same 10,000 steps again. This is what makes re-running this cell after a
    # disconnect the recovery path rather than a fresh start.
    # A finished run leaves a resume point behind (the last one is short of max_steps by
    # construction). Without this check, re-running the cell after a successful run would
    # resume from that stale point and retrain the last 10,000 steps for hours, and then
    # warm-start the chat stage from whichever final.npz sorted first.
    done = completed_final(out_alerts, MAX_STEPS)
    if done:
        print('alerts stage already completed at step %d - reusing %s'
              % (_manifest(done)['step'], done), flush=True)
        run_logged_succeeded = True
    else:
        run_logged_succeeded = False
    resume_from = None if done else find_resume_point(out_alerts)
    cmd = ('%s training/gpu_train.py '
           '--config configs/astra_distill_v4_alerts.json '
           '--out %s --cache-dir /tmp/astra_cache --dtype %s'
           % (sys.executable, out_alerts, DTYPE))
    if resume_from:
        print('resuming alerts from %s' % resume_from, flush=True)
        cmd += ' --resume %s' % resume_from
    elif not done:
        print('no resumable checkpoint - starting the alerts stage from step 0', flush=True)
    if not done:
        run_logged(cmd, 'train_alerts.log')
    alerts_final = completed_final(out_alerts, MAX_STEPS) or find_final(out_alerts)
    assert alerts_final, 'alerts stage produced no final.npz'
    print('alerts final:', alerts_final)
    persist(out_alerts)

    assert os.path.exists(alerts_final), 'alerts final.npz vanished - rerun alerts stage'
    out_chat = os.path.join(RUNS, 'distill_v4_chat')
    os.makedirs(out_chat, exist_ok=True)
    chat_done = completed_final(out_chat, MAX_STEPS)
    if chat_done:
        print('chat stage already completed at step %d - reusing %s'
              % (_manifest(chat_done)['step'], chat_done), flush=True)
    else:
        run_logged('%s training/gpu_train.py '
                   '--config configs/astra_distill_v4_chat.json '
                   '--out %s --resume %s --reset-step '
                   '--cache-dir /tmp/astra_cache --dtype %s'
                   % (sys.executable, out_chat, alerts_final, DTYPE), 'train_chat.log')
    chat_final = completed_final(out_chat, MAX_STEPS) or find_final(out_chat)
    assert chat_final, 'chat stage produced no final.npz'
    print('chat final:', chat_final)
    persist(chat_final and out_chat)
    print('\ndone - checkpoints under %s'
          % (DRIVE_ROOT if DRIVE_OK else '(no Drive, so only on this VM)'))

## Stage 5b — Smoke test

A checkpoint that loads is not a checkpoint that learned something. Three prompts and a
held-out alert row, so you can see the output shape before trusting it. An empty reply or a
string of special tokens means the tokenizer and the checkpoint disagree — a real bug, not a
quality problem.


In [ ]:
import os, sys, glob, json, subprocess

import numpy as np

REPO = '/content/astra'
if not os.path.isdir(os.path.join(REPO, 'tools')):
    subprocess.run('git clone --depth 1 https://github.com/anoneurx/astra.git %s' % REPO,
                   shell=True, check=True)
os.chdir(REPO)
sys.path.insert(0, os.path.join(REPO, 'python'))

from astra.inference import KVCache, decode
from astra.learning.evaluate import load_model
from astra.model import ModelConfig
from astra.tokenizer.word import WordLevel

candidates = sorted(glob.glob(os.path.join(RUNS, 'distill_v4_chat', '**', 'final.npz'), recursive=True))
assert candidates, 'no distilled chat checkpoint - set RUN_TRAIN = True and run Stage 6a'
CKPT = candidates[0]
print('checkpoint:', CKPT)

cfg = json.load(open('configs/astra_distill_v4_chat.json', encoding='utf-8'))
tok = WordLevel.load(cfg['tokenizer'])
mcfg = ModelConfig.from_dict(cfg['model'])
mcfg.vocab_size = len(tok)

model = load_model(mcfg, CKPT)
cache = KVCache(mcfg)
SPECIALS = set(range(tok.num_special))
history = []


def ask(user_text, max_new=64, temperature=0.7, top_k=40, rep_penalty=1.15):
    global history
    history.append('You: %s' % user_text)
    prompt = '\n'.join(history) + '\nAstra:'
    ids = tok.encode(prompt)[-(mcfg.max_seq_len - max_new):]
    new = decode(model, ids, max_new=max_new, temperature=temperature,
                 rng=np.random.default_rng(len(history)), top_k=top_k,
                 cache=cache, rep_penalty=rep_penalty, forbidden=SPECIALS)
    reply = tok.decode(new).split('\nYou:')[0].strip()
    history.append('Astra: %s' % reply)
    return reply


for q in ['an ssh login as root from 203.0.113.9 at 02:14 outside the change window',
          'explain what early warning means in a security alert stream',
          'what should an analyst do about a service account whose permissions widened']:
    print('You  :', q)
    print('Astra:', ask(q))
    print()

alert_path = 'datasets/distillation_v3/alerts/eval.txt'
if os.path.exists(alert_path):
    print('held-out alert row (never trained on):')
    print(' ', open(alert_path, encoding='utf-8').readline().strip()[:320])

## Stage 6 — Download

One zip, because the pieces are useless apart. The one people forget is the **tokenizer**:
`.gitignore` excludes `tokenizer/artifacts/`, so it is not in the repo and cannot be
re-cloned, and every config points at it. Without it nothing loads.

* `datasets/distillation_v3/**` — the corpora, `records.jsonl` (provenance) and `manifest.json`
  (every drop count, repair, the class-balance before/after, the leak report, preflight).
* `configs/astra_distill_v4_{chat,alerts}.json` — paths already point at the data above.
* `tokenizer/artifacts/prose_chat_word.json` — **required**, rebuilt with `--extra`.
* `runs/**/final.npz` — the checkpoints, only if `RUN_TRAIN` ran. This is the actual model.
* `runs/**/*.manifest.json` — step, seed, params, loss history. Not decoration:
  `load_checkpoint` reads them, and a warm-start without them silently restarts the schedule.

`SHA256SUMS.txt` is included so you can check the zip arrived intact.


In [ ]:
import glob, zipfile, hashlib

want = []
want += sorted(glob.glob(os.path.join(DATA_DIR, '**', '*.txt'), recursive=True))
want += [os.path.join(DATA_DIR, 'records.jsonl'), os.path.join(DATA_DIR, 'manifest.json')]
want += ['configs/astra_distill_v4_chat.json', 'configs/astra_distill_v4_alerts.json']
want += [os.path.join(REPO, TOK_ARTIFACT)]
want += sorted(glob.glob(os.path.join(RUNS, '**', 'final.npz'), recursive=True))
want += sorted(glob.glob(os.path.join(RUNS, '**', 'report.json'), recursive=True))
want += sorted(glob.glob(os.path.join(RUNS, '**', '*.manifest.json'), recursive=True))

have = [p for p in want if os.path.exists(p)]
skipped = [os.path.basename(p) for p in want if not os.path.exists(p)]
if skipped:
    print('not produced, skipping:', skipped)
assert have, 'nothing to package - did the generation cell run?'


def arcname(path):
    """The name this file has INSIDE the zip. Checkpoints sit under RUNS, which is outside
    the repo, so they cannot be laid out relative to the repo without climbing out with '..'
    - and SHA256SUMS.txt has to list exactly these names or `sha256sum -c` cannot verify."""
    if os.path.abspath(path).startswith(os.path.abspath(RUNS) + os.sep):
        return os.path.join('runs', os.path.relpath(path, RUNS))
    return os.path.relpath(path, REPO)


arcs = {p: arcname(p) for p in have}
OUT = os.path.abspath('astra_distill_v4.zip')
with zipfile.ZipFile(OUT, 'w', zipfile.ZIP_DEFLATED) as z:
    for p in have:
        z.write(p, arcs[p])
    z.writestr('SHA256SUMS.txt', '\n'.join(
        '%s  %s' % (hashlib.sha256(open(p, 'rb').read()).hexdigest(), arcs[p])
        for p in have) + '\n')

print('%-52s %10s' % ('file', 'bytes'))
for p in have:
    print('%-52s %10d' % (arcs[p], os.path.getsize(p)))
print('\n%d files, %.1f MB zipped' % (len(have), os.path.getsize(OUT) / 1e6))

if not DRIVE_OK:
    try:
        from google.colab import files
        print('\ndownloading', OUT)
        files.download(OUT)
    except ImportError:
        print('\nnot under Colab - the zip is at', OUT)
else:
    shutil.copy2(OUT, os.path.join(DRIVE_ROOT, os.path.basename(OUT)))
    print('\ncopied to Drive:', os.path.join(DRIVE_ROOT, os.path.basename(OUT)))